In [1]:
!pip install tamil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.9/208.9 kB 12.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import pandas as pd
import re
from itertools import zip_longest


In [4]:
# Tamil Unicode regex
TAMIL_REGEX = re.compile(r'^[\u0B80-\u0BFF]+$')

# English + optional Tamil suffix regex
ENGLISH_WITH_SUFFIX_REGEX = re.compile(r'^([A-Za-z]+)(-[\u0B80-\u0BFF]+)?$')

# Helper functions
def extract_english_root(word):
    match = ENGLISH_WITH_SUFFIX_REGEX.match(word)
    if match:
        return match.group(1)
    return None

def is_tamil_letters(word):
    return bool(TAMIL_REGEX.match(word))

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/vassista22_code_switching_adalora_checkpoints_phase1/final_predictions.csv")

In [ ]:
total_english_words = 0
english_recognized_in_tamil_count = 0
misrecognized_list = []

for idx, row in df.iterrows():
    ref_words = row["Reference_Text"].split()
    hyp_words = row["Generated_Text"].split()

    for r_word, h_word in zip_longest(ref_words, hyp_words, fillvalue=""):
        root = extract_english_root(r_word)
        if root:  # English word or English+suffix
            total_english_words += 1
            if is_tamil_letters(h_word):
                english_recognized_in_tamil_count += 1
                misrecognized_list.append([row["Reference_Text"], row["Generated_Text"], r_word, h_word])

# Save CSV
misrec_df = pd.DataFrame(
    misrecognized_list,
    columns=["Reference_Text", "Generated_Text", "English_WORD", "Tamil_HYP"]
)
misrec_df.to_csv("/content/drive/MyDrive/vassista22_code_switching_adalora_checkpoints_phase1/english_to_tamil_misrecognized.csv", index=False)

# Metrics
english_to_tamil_error_rate = english_recognized_in_tamil_count / total_english_words

print("=== English→Tamil Misrecognition Metrics ===")
print(f"Total English words in REF (including suffixes): {total_english_words}")
print(f"English words recognized in Tamil letters: {english_recognized_in_tamil_count}")
print(f"English→Tamil Misrecognition Rate: {english_to_tamil_error_rate:.4f}")
print("\nCSV saved:/content/drive/MyDrive/vassista22_code_switching_adalora_checkpoints_phase1/english_to_tamil_misrecognized.csv ")

=== English→Tamil Misrecognition Metrics ===
Total English words in REF (including suffixes): 12283
English words recognized in Tamil letters: 1102
English→Tamil Misrecognition Rate: 0.0897

CSV saved:/content/drive/MyDrive/vassista22_code_switching_adalora_checkpoints_phase1/english_to_tamil_misrecognized.csv 


# Levinstein

In [6]:
# Install dependencies

!pip install python-Levenshtein
!pip install jiwer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.8 MB/s eta 0:00:00


In [7]:


# -------------------------------
import pandas as pd
import re
from itertools import zip_longest
import Levenshtein

# Tamil Unicode regex
TAMIL_REGEX = re.compile(r'^[\u0B80-\u0BFF]+$')

# English + optional Tamil suffix regex
ENGLISH_WITH_SUFFIX_REGEX = re.compile(r'^([A-Za-z]+)(-[\u0B80-\u0BFF]+)?$')

# Helper functions
def extract_english_root(word):
    match = ENGLISH_WITH_SUFFIX_REGEX.match(word)
    if match:
        return match.group(1)
    return None

def is_tamil_letters(word):
    return bool(TAMIL_REGEX.match(word))

# Function to align REF and HYP words using Levenshtein distance
def align_words(ref_words, hyp_words):
    """
    Returns aligned REF and HYP lists using minimum edit distance
    """
    import numpy as np
    n = len(ref_words)
    m = len(hyp_words)
    dp = np.zeros((n+1, m+1), dtype=int)

    for i in range(n+1):
        dp[i][0] = i
    for j in range(m+1):
        dp[0][j] = j

    for i in range(1, n+1):
        for j in range(1, m+1):
            if ref_words[i-1] == hyp_words[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])

    # Backtrack
    aligned_ref = []
    aligned_hyp = []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref_words[i-1] == hyp_words[j-1]:
            aligned_ref.insert(0, ref_words[i-1])
            aligned_hyp.insert(0, hyp_words[j-1])
            i -= 1
            j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            aligned_ref.insert(0, ref_words[i-1])
            aligned_hyp.insert(0, hyp_words[j-1])
            i -= 1
            j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            aligned_ref.insert(0, ref_words[i-1])
            aligned_hyp.insert(0, "")
            i -= 1
        else:
            aligned_ref.insert(0, "")
            aligned_hyp.insert(0, hyp_words[j-1])
            j -= 1
    return aligned_ref, aligned_hyp

# -------------------------------
# Load CSV


In [12]:
df = pd.read_csv("/content/drive/MyDrive/vassista22_code_switching_adalora_checkpoints_phase1/final_predictions.csv")  # Columns: Reference_Text, Generated_Text





In [13]:
# English→Tamil misrecognition analysis
total_english_words = 0
english_recognized_in_tamil_count = 0
misrecognized_list = []

for idx, row in df.iterrows():
    ref_words = row["Reference_Text"].split()
    hyp_words = row["Generated_Text"].split()

    aligned_ref, aligned_hyp = align_words(ref_words, hyp_words)

    for r_word, h_word in zip(aligned_ref, aligned_hyp):
        root = extract_english_root(r_word)
        if root:
            total_english_words += 1
            if is_tamil_letters(h_word):
                english_recognized_in_tamil_count += 1
                misrecognized_list.append([row["Reference_Text"], row["Generated_Text"], r_word, h_word])


In [14]:
# -------------------------------
# Save CSV
misrec_df = pd.DataFrame(
    misrecognized_list,
    columns=["Reference_Text", "Generated_Text", "English_WORD", "Tamil_HYP"]
)
misrec_df.to_csv("/content/drive/MyDrive/vassista22_code_switching_adalora_checkpoints_phase1/misconfigured_2.csv", index=False)

# -------------------------------
# Metrics
english_to_tamil_error_rate = english_recognized_in_tamil_count / total_english_words

print("=== English→Tamil Misrecognition Metrics ===")
print(f"Total English words in REF (including suffixes): {total_english_words}")
print(f"English words recognized in Tamil letters: {english_recognized_in_tamil_count}")
print(f"English→Tamil Misrecognition Rate: {english_to_tamil_error_rate:.4f}")
print("\nCSV saved:")

=== English→Tamil Misrecognition Metrics ===
Total English words in REF (including suffixes): 12283
English words recognized in Tamil letters: 449
English→Tamil Misrecognition Rate: 0.0366

CSV saved:


# Experiment 2

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/vassista22_code_switching_adalora_checkpoints_phase1_experiment2/final_predictions.csv")  # Columns: Reference_Text, Generated_Text





In [ ]:
# English→Tamil misrecognition analysis
total_english_words = 0
english_recognized_in_tamil_count = 0
misrecognized_list = []

for idx, row in df.iterrows():
    ref_words = row["Reference_Text"].split()
    hyp_words = row["Generated_Text"].split()

    aligned_ref, aligned_hyp = align_words(ref_words, hyp_words)

    for r_word, h_word in zip(aligned_ref, aligned_hyp):
        root = extract_english_root(r_word)
        if root:
            total_english_words += 1
            if is_tamil_letters(h_word):
                english_recognized_in_tamil_count += 1
                misrecognized_list.append([row["Reference_Text"], row["Generated_Text"], r_word, h_word])


In [ ]:
# -------------------------------
# Save CSV
misrec_df = pd.DataFrame(
    misrecognized_list,
    columns=["Reference_Text", "Generated_Text", "English_WORD", "Tamil_HYP"]
)
misrec_df.to_csv("/content/drive/MyDrive/vassista22_code_switching_adalora_checkpoints_phase1_experiment2/misconfigured_2.csv", index=False)

# -------------------------------
# Metrics
english_to_tamil_error_rate = english_recognized_in_tamil_count / total_english_words

print("=== English→Tamil Misrecognition Metrics ===")
print(f"Total English words in REF (including suffixes): {total_english_words}")
print(f"English words recognized in Tamil letters: {english_recognized_in_tamil_count}")
print(f"English→Tamil Misrecognition Rate: {english_to_tamil_error_rate:.4f}")
print("\nCSV saved:")

=== English→Tamil Misrecognition Metrics ===
Total English words in REF (including suffixes): 12283
English words recognized in Tamil letters: 584
English→Tamil Misrecognition Rate: 0.0475

CSV saved:


# Experiment 3

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/vassista22_code_switching_adalora_checkpoints_phase1_experiment3/final_predictions.csv")  # Columns: Reference_Text, Generated_Text





In [ ]:
# English→Tamil misrecognition analysis
total_english_words = 0
english_recognized_in_tamil_count = 0
misrecognized_list = []

for idx, row in df.iterrows():
    ref_words = row["Reference_Text"].split()
    hyp_words = row["Generated_Text"].split()

    aligned_ref, aligned_hyp = align_words(ref_words, hyp_words)

    for r_word, h_word in zip(aligned_ref, aligned_hyp):
        root = extract_english_root(r_word)
        if root:
            total_english_words += 1
            if is_tamil_letters(h_word):
                english_recognized_in_tamil_count += 1
                misrecognized_list.append([row["Reference_Text"], row["Generated_Text"], r_word, h_word])


In [ ]:
# -------------------------------
# Save CSV
misrec_df = pd.DataFrame(
    misrecognized_list,
    columns=["Reference_Text", "Generated_Text", "English_WORD", "Tamil_HYP"]
)
misrec_df.to_csv("/content/drive/MyDrive/vassista22_code_switching_adalora_checkpoints_phase1_experiment3/misconfigured_2.csv", index=False)

# -------------------------------
# Metrics
english_to_tamil_error_rate = english_recognized_in_tamil_count / total_english_words

print("=== English→Tamil Misrecognition Metrics ===")
print(f"Total English words in REF (including suffixes): {total_english_words}")
print(f"English words recognized in Tamil letters: {english_recognized_in_tamil_count}")
print(f"English→Tamil Misrecognition Rate: {english_to_tamil_error_rate:.4f}")
print("\nCSV saved:")

=== English→Tamil Misrecognition Metrics ===
Total English words in REF (including suffixes): 12283
English words recognized in Tamil letters: 453
English→Tamil Misrecognition Rate: 0.0369

CSV saved:


# Experiment 4

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/vassista22_code_switching_adalora_checkpoints_phase1_experiment4/final_predictions.csv")  # Columns: Reference_Text, Generated_Text


In [ ]:
# English→Tamil misrecognition analysis
total_english_words = 0
english_recognized_in_tamil_count = 0
misrecognized_list = []

for idx, row in df.iterrows():
    ref_words = row["Reference_Text"].split()
    hyp_words = row["Generated_Text"].split()

    aligned_ref, aligned_hyp = align_words(ref_words, hyp_words)

    for r_word, h_word in zip(aligned_ref, aligned_hyp):
        root = extract_english_root(r_word)
        if root:
            total_english_words += 1
            if is_tamil_letters(h_word):
                english_recognized_in_tamil_count += 1
                misrecognized_list.append([row["Reference_Text"], row["Generated_Text"], r_word, h_word])


In [ ]:
# -------------------------------
# Save CSV
misrec_df = pd.DataFrame(
    misrecognized_list,
    columns=["Reference_Text", "Generated_Text", "English_WORD", "Tamil_HYP"]
)
misrec_df.to_csv("/content/drive/MyDrive/vassista22_code_switching_adalora_checkpoints_phase1_experiment4/misconfigured_2.csv", index=False)

# -------------------------------
# Metrics
english_to_tamil_error_rate = english_recognized_in_tamil_count / total_english_words

print("=== English→Tamil Misrecognition Metrics ===")
print(f"Total English words in REF (including suffixes): {total_english_words}")
print(f"English words recognized in Tamil letters: {english_recognized_in_tamil_count}")
print(f"English→Tamil Misrecognition Rate: {english_to_tamil_error_rate:.4f}")
print("\nCSV saved:")

=== English→Tamil Misrecognition Metrics ===
Total English words in REF (including suffixes): 4779
English words recognized in Tamil letters: 174
English→Tamil Misrecognition Rate: 0.0364

CSV saved:


# Experiment 5

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/vassista22_code_switching_adalora_checkpoints_phase1_experiment6/final_predictions.csv")  # Columns: Reference_Text, Generated_Text


In [ ]:
# English→Tamil misrecognition analysis
total_english_words = 0
english_recognized_in_tamil_count = 0
misrecognized_list = []

for idx, row in df.iterrows():
    ref_words = row["Reference_Text"].split()
    hyp_words = row["Generated_Text"].split()

    aligned_ref, aligned_hyp = align_words(ref_words, hyp_words)

    for r_word, h_word in zip(aligned_ref, aligned_hyp):
        root = extract_english_root(r_word)
        if root:
            total_english_words += 1
            if is_tamil_letters(h_word):
                english_recognized_in_tamil_count += 1
                misrecognized_list.append([row["Reference_Text"], row["Generated_Text"], r_word, h_word])


In [ ]:
# -------------------------------
# Save CSV
misrec_df = pd.DataFrame(
    misrecognized_list,
    columns=["Reference_Text", "Generated_Text", "English_WORD", "Tamil_HYP"]
)
misrec_df.to_csv("/content/drive/MyDrive/vassista22_code_switching_adalora_checkpoints_phase1_experiment6/misconfigured_2.csv", index=False)

# -------------------------------
# Metrics
english_to_tamil_error_rate = english_recognized_in_tamil_count / total_english_words

print("=== English→Tamil Misrecognition Metrics ===")
print(f"Total English words in REF (including suffixes): {total_english_words}")
print(f"English words recognized in Tamil letters: {english_recognized_in_tamil_count}")
print(f"English→Tamil Misrecognition Rate: {english_to_tamil_error_rate:.4f}")
print("\nCSV saved:")

=== English→Tamil Misrecognition Metrics ===
Total English words in REF (including suffixes): 4779
English words recognized in Tamil letters: 181
English→Tamil Misrecognition Rate: 0.0379

CSV saved:


# Whisper

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Whisper_large_v3_code_switching_adalora_checkpoints_phase1/final_predictions.csv")  # Columns: Reference_Text, Generated_Text


In [ ]:
# English→Tamil misrecognition analysis
total_english_words = 0
english_recognized_in_tamil_count = 0
misrecognized_list = []

for idx, row in df.iterrows():
    ref_words = row["Reference_Text"].split()
    hyp_words = row["Generated_Text"].split()

    aligned_ref, aligned_hyp = align_words(ref_words, hyp_words)

    for r_word, h_word in zip(aligned_ref, aligned_hyp):
        root = extract_english_root(r_word)
        if root:
            total_english_words += 1
            if is_tamil_letters(h_word):
                english_recognized_in_tamil_count += 1
                misrecognized_list.append([row["Reference_Text"], row["Generated_Text"], r_word, h_word])


In [ ]:
# -------------------------------
# Save CSV
misrec_df = pd.DataFrame(
    misrecognized_list,
    columns=["Reference_Text", "Generated_Text", "English_WORD", "Tamil_HYP"]
)
misrec_df.to_csv("/content/drive/MyDrive/Whisper_large_v3_code_switching_adalora_checkpoints_phase1/misconfigured_2.csv", index=False)

# -------------------------------
# Metrics
english_to_tamil_error_rate = english_recognized_in_tamil_count / total_english_words

print("=== English→Tamil Misrecognition Metrics ===")
print(f"Total English words in REF (including suffixes): {total_english_words}")
print(f"English words recognized in Tamil letters: {english_recognized_in_tamil_count}")
print(f"English→Tamil Misrecognition Rate: {english_to_tamil_error_rate:.4f}")
print("\nCSV saved:")

=== English→Tamil Misrecognition Metrics ===
Total English words in REF (including suffixes): 12283
English words recognized in Tamil letters: 436
English→Tamil Misrecognition Rate: 0.0355

CSV saved:


# Zeroshot whisper

In [15]:
df = pd.read_csv("/content/drive/MyDrive/output_metrics_whisper_v3_large_default_state.csv")


In [16]:
# English→Tamil misrecognition analysis
total_english_words = 0
english_recognized_in_tamil_count = 0
misrecognized_list = []

for idx, row in df.iterrows():
    ref_words = row["script_text"].split()
    hyp_words = row["transcription"].split()

    aligned_ref, aligned_hyp = align_words(ref_words, hyp_words)

    for r_word, h_word in zip(aligned_ref, aligned_hyp):
        root = extract_english_root(r_word)
        if root:
            total_english_words += 1
            if is_tamil_letters(h_word):
                english_recognized_in_tamil_count += 1
                misrecognized_list.append([row["script_text"], row["transcription"], r_word, h_word])


In [18]:
# -------------------------------
# Save CSV
misrec_df = pd.DataFrame(
    misrecognized_list,
    columns=["script_text", "transcription", "English_WORD", "Tamil_HYP"]
)
misrec_df.to_csv("/content/drive/MyDrive/whisper_zeroshot_misconfigured_2.csv", index=False)

# -------------------------------
# Metrics
english_to_tamil_error_rate = english_recognized_in_tamil_count / total_english_words

print("=== English→Tamil Misrecognition Metrics ===")
print(f"Total English words in REF (including suffixes): {total_english_words}")
print(f"English words recognized in Tamil letters: {english_recognized_in_tamil_count}")
print(f"English→Tamil Misrecognition Rate: {english_to_tamil_error_rate:.4f}")
print("\nCSV saved:")

=== English→Tamil Misrecognition Metrics ===
Total English words in REF (including suffixes): 12695
English words recognized in Tamil letters: 7093
English→Tamil Misrecognition Rate: 0.5587

CSV saved:


# Zeroshot vassista 22 large

In [20]:
df = pd.read_csv("/content/drive/MyDrive/output_metrics_set_to_default_vasista22_large_v2.csv")


In [21]:
# English→Tamil misrecognition analysis
total_english_words = 0
english_recognized_in_tamil_count = 0
misrecognized_list = []

for idx, row in df.iterrows():
    ref_words = row["script_text"].split()
    hyp_words = row["transcription"].split()

    aligned_ref, aligned_hyp = align_words(ref_words, hyp_words)

    for r_word, h_word in zip(aligned_ref, aligned_hyp):
        root = extract_english_root(r_word)
        if root:
            total_english_words += 1
            if is_tamil_letters(h_word):
                english_recognized_in_tamil_count += 1
                misrecognized_list.append([row["script_text"], row["transcription"], r_word, h_word])


In [22]:
# -------------------------------
# Save CSV
misrec_df = pd.DataFrame(
    misrecognized_list,
    columns=["script_text", "transcription", "English_WORD", "Tamil_HYP"]
)
misrec_df.to_csv("/content/drive/MyDrive/vassista_zeroshot_misconfigured_2.csv", index=False)

# -------------------------------
# Metrics
english_to_tamil_error_rate = english_recognized_in_tamil_count / total_english_words

print("=== English→Tamil Misrecognition Metrics ===")
print(f"Total English words in REF (including suffixes): {total_english_words}")
print(f"English words recognized in Tamil letters: {english_recognized_in_tamil_count}")
print(f"English→Tamil Misrecognition Rate: {english_to_tamil_error_rate:.4f}")
print("\nCSV saved:")

=== English→Tamil Misrecognition Metrics ===
Total English words in REF (including suffixes): 12695
English words recognized in Tamil letters: 11097
English→Tamil Misrecognition Rate: 0.8741

CSV saved:
